In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [3]:
! pip install transformers datasets sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 80.6 MB/s eta 0:00:00:00:0100:01


In [4]:
from datasets import load_dataset

squad=load_dataset("squad")
sciq=load_dataset("sciq")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/87599 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10570 [00:00<?, ? examples/s]

...

data/validation-00000-of-00001.parquet:   0%|          | 0.00/339k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/343k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/11679 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [5]:
print(squad)

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 10570
    })
})


In [6]:
print(sciq)

DatasetDict({
    train: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 11679
    })
    validation: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['question', 'distractor3', 'distractor1', 'distractor2', 'correct_answer', 'support'],
        num_rows: 1000
    })
})


In [7]:
def extract_squad_text(squad):
    texts=[]

    for item in squad:
        text=item['context']
        if text != '':
            texts.append(text)

    return texts

In [8]:
squad_texts=extract_squad_text(squad['train'])

In [9]:
def extract_sciq_texts(sciq):
    texts=[]

    for item in sciq:
        text=item['support']
        if text != '':
            texts.append(text)

    return texts

In [10]:
sciq_texts=extract_sciq_texts(sciq['train'])

In [11]:
all_texts=squad_texts+sciq_texts

In [12]:
print(len(all_texts))

98080


In [13]:
def preprocess(text):
    return text.strip().replace('\n',' ')

def chunk_text(text,chunk_size=200):
    words=text.split()
    chunked_texts=[]

    for i in range(0,len(words),chunk_size):
        chunk=" ".join(words[i:i+chunk_size])
        chunked_texts.append(chunk)

    return chunked_texts

In [14]:
processed_chunks=[]

for text in all_texts:
    preprocessed_text=preprocess(text)
    chunked_texts=chunk_text(text)
    processed_chunks.extend(chunked_texts)

In [15]:
processed_chunks=list(set(processed_chunks)) # to keep unique texts only

In [16]:
print(all_texts[0])
print(processed_chunks[0])

Architecturally, the school has a Catholic character. Atop the Main Building's gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.
The scientific method is not a step by step, linear process. It is a way of learning about the world through the application of knowledge. Scientists must be able to have an idea of what the answer to an investigation should be. In order for scientists to make educated guesses about the answers, they wi

In [17]:
from sentence_transformers import SentenceTransformer

embedder=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
# embedder=SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
embeddings=embedder.encode(processed_chunks,show_progress_bar=True)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/979 [00:00<?, ?it/s]

In [19]:
print(embeddings.shape)

(31314, 384)


In [21]:
import numpy as np
import faiss

index=faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

In [22]:
print(type(index))

<class 'faiss.swigfaiss.IndexFlatL2'>


In [24]:
def retrieve(query,k=3):
    query_vec=embedder.encode(query).reshape(1,-1).astype("float32")

    distances,indices=index.search(np.array(query_vec),k)
    retrieved_texts=[processed_chunks[i] for i in indices[0]]

    return retrieved_texts

In [25]:
retrieved_texts=retrieve("what is a mitochondria?",3)

In [26]:
print(retrieved_texts)

['Mitochondria are organelles whose membranes are specialized for aerobic respiration.', 'The mitochondria are the powerhouses of the cell. Mitochondria are the organelles where cellular energy is produced, providing the energy needed to power chemical reactions. This process, known as cellular respiration , produces energy is in the form of ATP (adenosine triphosphate). Cells that use a lot of energy may have thousands of mitochondria.', 'Mitochondria are thought to have evolved from ancient prokaryotic cells.']


In [27]:
import torch

In [24]:
# from transformers import T5Tokenizer, T5ForConditionalGeneration

# tokenizer=T5Tokenizer.from_pretrained("google/flan-t5-base")
# model=T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
# tokenizer=T5Tokenizer.from_pretrained("google/flan-t5-large")
# model=T5ForConditionalGeneration.from_pretrained("google/flan-t5-large")
# tokenizer=T5Tokenizer.from_pretrained("google/flan-t5-xl")
# model=T5ForConditionalGeneration.from_pretrained("google/flan-t5-xl",device_map="auto",torch_dtype=torch.float16)

In [28]:
device="cuda:0" if torch.cuda.is_available() else "cpu"

In [29]:
pip install -U bitsandbytes>=0.46.1

Note: you may need to restart the kernel to use updated packages.


In [30]:
from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig

model_id="/kaggle/input/models/google/gemma-2/transformers/gemma-2-9b-it/2"

quantization_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

gemma_tokenizer=AutoTokenizer.from_pretrained(model_id)
gemma_model=AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=quantization_config
)

Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

In [70]:
# import os
# os.remove("/kaggle/working/data.csv")

In [72]:
import numpy as np

BATCH_SIZE=50
NUM_BATCHES=1

intents = {
    0: {
        "name": "Definition",
        "description": "student asking to define something or what something means"
    },

    1: {
        "name": "Explanation",
        "description": "student asking to explain how or why a scientific concept works"
    },

    2: {
        "name": "Give Examples",
        "description": "student asking for examples of a scientific concept"
    },

    3: {
        "name": "Comparison",
        "description": "student asking similarities or differences between concepts"
    },

    4: {
        "name": "Categorization",
        "description": "student asking to classify or group scientific things"
    },

    5: {
        "name": "Give Practice Questions",
        "description": "student asking for quizzes, MCQs, exercises, worksheets, practice problems, or numericals"
    },

    6: {
        "name": "General Chat",
        "description": "casual non-science conversation"
    }
}

dataset=[]

for label,intent in intents.items():
    for i in range(NUM_BATCHES):
        prompt = f"""Generate 30 diverse examples(MUST be BASED ON SCIENCE TOPICS) of a SCIENCE STUDENT asking for a question based on {intent}. 
    Output them as a simple list, one per line, no numbering."""
    
    
        inputs=gemma_tokenizer(prompt,return_tensors="pt")
        input_len=inputs['input_ids'].shape[1]
        outputs=gemma_model.generate(**inputs,max_new_tokens=1000)
        examples=gemma_tokenizer.decode(outputs[0][input_len:],skip_special_tokens=True)
        for data in examples.split("\n"):
            if data!="":
                dataset.append({"text": data, "label": label})

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


In [ ]:
prompt = f"""Generate 30 diverse examples(which MUST BE BASED ON SCIENCE TOPICS) of a SCIENCE STUDENT asking for a question which MUST be explicitly based on {intent}. 
    Output them as a simple list, one per line, no numbering."""

In [ ]:
prompt = f"""Generate 30 diverse examples(which MUST BE EXPLICITLY BASED ON SCIENCE TOPICS) of a SCIENCE STUDENT asking for a question which MUST be based on {intent}. 
        Output them as a simple list, one per line, no numbering."""

In [ ]:
# data good

prompt = f"""Generate 20 diverse examples of a SCIENCE STUDENT asking for a question based on {intent}. 
    Output them as a simple list, one per line, no numbering."""

In [ ]:
prompt = f"""Generate 20 diverse examples of a SCIENCE STUDENT asking for a question based on {intent}. 
    Output them as a simple list, one per line, no numbering.
    
    Examples(for each class): What is gravity? 
    Explain how does photosynthesis work?
    Give me an example of a cell organelle
    How does the structure of plant cells differ from animal cells?
    How can matter be categorized into elements, compounds, and mixtures?
    Provide one practice question on Quantum physics.
    How to improve my efficiency?"""

In [73]:
# if data already created
dataset=pd.read_csv('/kaggle/working/data.csv').to_dict(orient='records')
print(dataset)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/data.csv'

In [74]:
print(dataset)

[{'text': 'Could you define the term "quantum entanglement"?', 'label': 0}, {'text': 'What does the term "photosynthesis" mean in the context of plant biology?', 'label': 0}, {'text': 'Can you define the concept of "natural selection" in evolutionary biology?', 'label': 0}, {'text': 'What does "homeostasis" refer to in physiology?', 'label': 0}, {'text': 'Could you define the term "isotope" in chemistry?', 'label': 0}, {'text': 'What does "gene expression" mean in molecular biology?', 'label': 0}, {'text': 'Can you define the concept of "gravity" in physics?', 'label': 0}, {'text': 'What does "biodiversity" refer to in ecology?', 'label': 0}, {'text': 'Could you define the term "mitochondria" in cell biology?', 'label': 0}, {'text': 'What does "climate change" mean in environmental science?', 'label': 0}, {'text': 'Can you define the concept of "plate tectonics" in geology?', 'label': 0}, {'text': 'What does "DNA replication" mean in genetics?', 'label': 0}, {'text': 'Could you define 

In [75]:
#postprocess

for item in dataset:
    if item['label']==2 or item['label']==4:
        item['text']=item['text'].split("and")[0]
    if item['label']==5:
        item['text']=item['text'].split(":")[1]

In [76]:
definition_examples = [
    "What is photosynthesis?",
    "Define gravity.",
    "What does evaporation mean?",
    "Can you define an atom?",
    "What is a chemical reaction?",
    "What is meant by an ecosystem?",
    "Define inertia.",
    "What exactly is osmosis?",
    "Define velocity.",
    "What is the periodic table?",
    "What does biodiversity mean?",
    "What is refraction of light?",
    "Define an acid.",
    "What is a base?",
    "What is kinetic energy?",
    "Define potential energy.",
    "What is DNA?",
    "What does pH mean?",
    "What is a solution in chemistry?",
    "Define force.",
    "What is pressure?",
    "Define a wave in physics.",
    "What is sound?",
    "What is light energy?",
    "Define species.",
    "What is a chemical bond?",
    "What is combustion?",
    "Define oxidation.",
    "What is respiration in biology?",
    "Define momentum.",
    "What is temperature?",
    "Define heat in physics.",
    "What is a neuron?",
    "What is the circulatory system?",
    "What are cell organelles?",
    "Define a proton.",
    "Define a neutron.",
    "What is an element?",
    "What is a compound?",
    "Define air pollution.",
    "What is climate change?",
    "Define a satellite.",
    "What is the solar system?",
    "What is a black hole?",
    "Define renewable energy.",
    "What does mutation mean?",
    "Define electric current.",
    "What is magnetism?",
    "Define friction.",
    "What is a galaxy?"
]

for chat in definition_examples:
    dataset.append({'text': chat, 'label': 0})

In [77]:
explanation_examples = [
    "Why does photosynthesis occur in plants?",
    "How does photosynthesis convert sunlight into energy?",
    "Explain what photosynthesis is and how it works.",
    "Why do objects fall due to gravity?",
    "How does gravity act between two bodies?",
    "Explain what gravity is and why it pulls objects downward.",
    "Why does ice float on water?",
    "How does evaporation happen in liquids?",
    "Explain what evaporation is and how it occurs.",
    "Why does a rainbow form after rain?",
    "How does refraction bend light in water?",
    "Explain what refraction of light is and how it works.",
    "Why do we hear thunder after lightning?",
    "How does sound travel through air?",
    "Explain what sound waves are and how they propagate.",
    "Why does metal expand when heated?",
    "How does osmosis occur across a membrane?",
    "Explain what osmosis is and how it works in cells.",
    "Why do humans need oxygen to survive?",
    "How does the heart pump blood through the body?",
    "Explain what the circulatory system is and how it functions.",
    "Why does rust form on iron objects?",
    "How does a battery generate electricity?",
    "Explain what electric current is and how it flows in a circuit.",
    "Why do planets revolve around the sun?",
    "How does inertia keep objects in motion?",
    "Explain what inertia is and how it affects motion.",
    "Why does friction produce heat?",
    "How does digestion break down food in the body?",
    "Explain what digestion is and how it works.",
    "Why do leaves change color during autumn?",
    "How does a magnet attract certain materials?",
    "Explain what magnetism is and how it works.",
    "Why does water boil at different temperatures at high altitudes?",
    "How does the water cycle operate in nature?",
    "Explain what the water cycle is and how it works.",
    "Why do stars appear to twinkle in the sky?",
    "How does electricity flow through a wire?",
    "Explain what electricity is and how it works.",
    "Why do acids react with bases?",
    "How does a rocket launch into space?",
    "Explain what thrust is and how rockets use it.",
    "Why do shadows change length during the day?",
    "How does DNA control traits in organisms?",
    "Explain what DNA is and how it determines characteristics.",
    "Why do gases expand when heated?",
    "How does the nervous system send signals?",
    "Explain what the nervous system is and how it works.",
    "Why does sound not travel in space?",
    "Explain why the Moon goes through different phases.",
    "Explain how vaccines help protect people from diseases.",
    "Explain the process by which clouds are formed.",
    "Explain the role of chlorophyll in green plants.",
    "Explain the differences between vertebrates and invertebrates.",
    "Explain the function of red blood cells in the human body.",
    "Explain the stages involved in the life cycle of a butterfly.",
    "Explain why seasons change throughout the year.",
    "Explain the importance of bees in pollination.",
    "Explain the process that causes eclipses to occur.",
    "Explain the purpose of the ozone layer in Earth's atmosphere.",
    "Explain how fossils provide evidence about prehistoric life.",
    "Explain the characteristics that distinguish mammals from other animals.",
    "Explain the mechanism behind the formation of ocean tides.",
    "Explain the structure and function of plant cells.",
    "Explain the differences between weather and climate.",
    "Explain the journey of water from the roots to the leaves in plants.",
    "Explain the significance of biodiversity in maintaining ecosystems.",
    "Explain the process through which bacteria reproduce.",
    "Explain the reasons why recycling benefits the environment."
    "Explain how the human eye focuses on objects at different distances.",
    "Explain why some materials conduct electricity better than others.",
    "Explain the process that plants use to reproduce.",
    "Explain how antibiotics work against bacterial infections.",
    "Explain the role of fungi in breaking down dead organisms.",
    "Explain why the sky appears blue during the day.",
    "Explain the relationship between force, mass, and acceleration.",
    "Explain how coral reefs are formed over time.",
    "Explain the purpose of roots, stems, and leaves in plants.",
    "Explain why some animals hibernate during winter.",
    "Explain the process through which sedimentary rocks are created.",
    "Explain how greenhouse gases influence Earth's temperature.",
    "Explain the role of enzymes in speeding up chemical reactions.",
    "Explain why different elements have unique chemical properties.",
    "Explain the movement of tectonic plates beneath Earth's surface.",
    "Explain how renewable resources differ from non-renewable resources.",
    "Explain the importance of the nitrogen cycle for living organisms.",
    "Explain how the human body maintains a stable internal temperature.",
    "Explain the process by which stars are born and eventually die.",
    "Explain why viruses require host cells in order to reproduce."
]

for chat in explanation_examples:
    dataset.append({'text': chat, 'label': 1})

In [51]:
print(dataset[-1])

{'text': 'Explain the reasons why recycling benefits the environment.', 'label': 1}


In [79]:
comparison_examples = [
     "What is the difference between photosynthesis and respiration?",
    "Compare acids and bases.",
    "What are the differences between mitosis and meiosis?",
    "How is velocity different from speed?",
    "Compare renewable and non-renewable energy sources.",
    "What is the difference between physical and chemical changes?",
    "How are plant cells and animal cells different?",
    "Compare evaporation and boiling.",
    "What is the difference between prokaryotic and eukaryotic cells?",
    "How are DNA and RNA different?",
    "Compare solids, liquids, and gases.",
    "What is the difference between conductors and insulators?",
    "How do heat and temperature differ?",
    "Compare natural selection and artificial selection.",
    "What is the difference between mass and weight?",
    "How are elements and compounds different?",
    "Compare series and parallel circuits.",
    "What is the difference between light waves and sound waves?",
    "How are arteries and veins different?",
    "Compare plant cells with animal cells.",
    "What is the difference between weather and climate?",
    "How are fossils different from rocks?",
    "Compare friction and gravity.",
    "What is the difference between speed and acceleration?",
    "How are vertebrates and invertebrates different?",
    "Compare acids and salts.",
    "What is the difference between kinetic and potential energy?",
    "How are physical properties different from chemical properties?",
    "Compare ionic bonds and covalent bonds.",
    "What is the difference between nucleus and cytoplasm?",
    "How are respiration and photosynthesis different?",
    "Compare magnetic and non-magnetic materials.",
    "What is the difference between evaporation and condensation?",
    "How are solids and liquids different in structure?",
    "Compare plants and fungi.",
    "What is the difference between ultrasound and audible sound?",
    "How are unicellular and multicellular organisms different?",
    "Compare reflection and refraction of light.",
    "What is the difference between a comet and an asteroid?",
    "How are enzymes different from hormones?",
    "Compare diffusion and osmosis.",
    "What is the difference between force and pressure?",
    "How are the nervous system and endocrine system different?",
    "Compare terrestrial and aquatic ecosystems.",
    "What is the difference between oxidation and reduction?",
    "How are mammals different from reptiles?",
    "Compare simple machines and complex machines.",
    "What is the difference between acid rain and normal rain?",
    "How are earthquakes different from volcanoes?",
    "Compare solar energy and wind energy."
]

for chat in comparison_examples:
    dataset.append({'text': chat, 'label': 3})

In [80]:
categorization_examples = [
    "What are the different types of rocks?",
    "What are the main categories of animals?",
    "What types of galaxies exist in the universe?",
    "What are the different classes of vertebrates?",
    "What kinds of chemical bonds are there?",
    "What are the major types of ecosystems?",
    "What are the different forms of energy?",
    "What types of volcanoes are there?",
    "What are the different kinds of cells?",
    "What are the categories of electromagnetic waves?",
    "Classify the following animals into mammals, reptiles, birds, amphibians, and fish.",
    "Group these elements into metals, non-metals, and metalloids.",
    "Sort the given planets into terrestrial planets and gas giants.",
    "Categorize the following organisms as producers, consumers, or decomposers.",
    "Separate these substances into acids, bases, and salts.",
    "Classify the following rocks as igneous, sedimentary, or metamorphic.",
    "Group the animals into vertebrates and invertebrates.",
    "Categorize these energy sources as renewable or non-renewable.",
    "Sort the following cells into prokaryotic and eukaryotic cells.",
    "Classify these waves as transverse or longitudinal waves.",
    "Group the following plants into monocots and dicots.",
    "Separate the organisms into autotrophs and heterotrophs.",
    "Classify the following forces as contact and non-contact forces.",
    "Group these compounds into organic and inorganic compounds.",
    "Categorize the following animals as herbivores, carnivores, or omnivores.",
    "Sort the given materials into conductors, insulators, and semiconductors.",
    "Classify these ecosystems as terrestrial and aquatic ecosystems.",
    "Group the following resources into biotic and abiotic resources.",
    "Categorize these reactions as exothermic and endothermic reactions.",
    "Separate the following mixtures into homogeneous and heterogeneous mixtures.",
    "Classify these blood vessels as arteries, veins, and capillaries.",
    "Group the following organisms into unicellular and multicellular organisms.",
    "Categorize these stars into dwarf stars, giant stars, and supergiants.",
    "Sort the following diseases into viral, bacterial, and fungal diseases.",
    "Classify these volcanoes as active, dormant, and extinct.",
    "Group the following clouds into cirrus, cumulus, and stratus clouds.",
    "Categorize these soils as sandy, clayey, and loamy soils.",
    "Separate the following particles into fermions and bosons.",
    "Classify the following bonds as ionic, covalent, and metallic bonds.",
    "Group these animals into cold-blooded and warm-blooded animals.",
    "Categorize the following circuits as series and parallel circuits.",
    "Sort these minerals into metallic and non-metallic minerals.",
    "Classify the following organisms into fungi, bacteria, protozoa, and algae.",
    "Group these resources into natural and man-made resources.",
    "Categorize the following pollutants as air, water, soil, and noise pollutants.",
    "Separate these plants into flowering and non-flowering plants.",
    "Classify the following galaxies as spiral, elliptical, and irregular galaxies.",
    "Group these fuels into fossil fuels and biofuels.",
    "Categorize the following hormones as steroid and peptide hormones.",
    "Sort these animals into endangered, vulnerable, and extinct species.",
    "Classify the following electromagnetic waves into radio waves, microwaves, infrared, visible light, ultraviolet, X-rays, and gamma rays.",
    "Group the following substances into solids, liquids, gases, and plasma.",
    "Categorize these microbes into useful and harmful microorganisms.",
    "Separate the following reactions into physical and chemical changes.",
    "Classify these nutrients into carbohydrates, proteins, fats, vitamins, and minerals.",
    "Group the following organisms into parasites, saprophytes, and symbionts.",
    "Categorize these enzymes into digestive and metabolic enzymes.",
    "Sort the following materials into biodegradable and non-biodegradable materials.",
    "Classify these celestial bodies into stars, planets, asteroids, and comets."
]

for chat in categorization_examples:
    dataset.append({'text': chat, 'label': 4})

In [81]:
practice_questions_examples = [
    "Give me 10 MCQs on Newton's laws of motion.",
    "Can you provide practice questions on chemical bonding?",
    "I want some numericals based on Ohm's law.",
    "Generate a quiz on the human digestive system.",
    "Give me worksheet questions on photosynthesis.",
    "Can you create practice problems on speed, velocity, and acceleration?",
    "Provide some MCQs on acids, bases, and salts.",
    "Give me difficult numericals on gravitation.",
    "Can you make a short test on cell structure and function?",
    "I need practice exercises on balancing chemical equations.",
    "Generate 5 assertion-reason questions on electricity.",
    "Can you give me some objective questions on the periodic table?",
    "Provide practice numericals for work, power, and energy.",
    "Give me a quiz about the solar system.",
    "Can you create worksheet questions on heredity and genetics?",
    "I want some case-based questions on ecosystems.",
    "Generate MCQs on heat and thermodynamics.",
    "Can you provide practice problems on current electricity?",
    "Give me some numerical questions on lenses and mirrors.",
    "Create a worksheet on respiration in organisms.",
    "Can you give me chapter-wise practice questions for atoms and molecules?",
    "Provide some HOTS questions on force and motion.",
    "Give me practice exercises on sound waves.",
    "Can you generate a mock test on magnetism?",
    "I need MCQs on metals and non-metals.",
    "Give me some application-based questions on friction.",
    "Can you provide numericals related to pressure?",
    "Generate a practice worksheet on reproduction in plants.",
    "Give me 20 quiz questions on environmental science.",
    "Can you make some diagram-based questions on the human heart?",
    "Provide practice questions on refraction of light.",
    "Give me MCQs on microorganisms.",
    "Can you create exercises on states of matter?",
    "I want numerical problems on momentum and collisions.",
    "Generate some true or false questions on nutrition in animals.",
    "Give me revision questions for the chapter on electricity.",
    "Can you provide worksheet problems on carbon compounds?",
    "Create a practice test on the nervous system.",
    "Give me some competency-based questions on climate change.",
    "Can you generate MCQs on waves and sound?",
    "Provide some sample questions on biodiversity.",
    "Give me exercises on atomic structure.",
    "Can you create practice numericals on kinetic energy?",
    "Generate quiz questions on biotechnology.",
    "I need some practice questions on reflection of light.",
    "Give me advanced MCQs on semiconductor devices.",
    "Can you provide practice worksheets on ecosystems and food chains?",
    "Generate some conceptual questions on electromagnetic induction.",
    "Give me numerical exercises on calorimetry.",
    "Can you make a science quiz on human diseases and immunity?"
]

for chat in practice_questions_examples:
    dataset.append({'text': chat, 'label': 5})

In [82]:
general_chat_examples = [
    "Hi, how are you?",
    "What are you doing right now?",
    "Can we just chat for a while?",
    "Tell me something interesting.",
    "I'm feeling bored today.",
    "Do you like music?",
    "What's your favorite movie genre?",
    "Can you recommend a good web series?",
    "What do people usually do on weekends?",
    "Tell me a funny joke.",
    "How was your day?",
    "Do you play games?",
    "What's your favorite food?",
    "Can you suggest a good hobby?",
    "I had a long day at school.",
    "Do you think AI will replace humans?",
    "What's the weather usually like in Delhi during summer?",
    "Can you motivate me a little?",
    "I don't feel like studying today.",
    "What are some good productivity tips?",
    "Can we talk about cricket?",
    "Who is your favorite fictional character?",
    "Suggest a good anime to watch.",
    "What should I do during holidays?",
    "Tell me a random fact.",
    "Do you think time travel is possible?",
    "What's the best way to relax?",
    "Can you suggest some good songs?",
    "I'm excited for my vacation.",
    "How can I improve my communication skills?",
    "What makes someone a good leader?",
    "Do you know any brain teasers?",
    "What's your opinion on social media?",
    "Can you tell me a short story?",
    "Why do people procrastinate so much?",
    "What are some fun things to do with friends?",
    "Can you recommend a good book?",
    "I feel nervous before exams.",
    "What's your favorite season?",
    "How do people stay consistent with habits?",
    "Can you suggest some healthy snacks?",
    "What are some popular careers these days?",
    "Do you think aliens exist?",
    "What is the meaning of success according to you?",
    "Can you help me plan my daily routine?",
    "What's a good way to start the morning?",
    "Why do people enjoy traveling?",
    "Let's talk about football.",
    "What are some useful life skills?",
    "Can you guess what mood I'm in?"
]

for chat in general_chat_examples:
    dataset.append({'text': chat, 'label': 6})

In [83]:
print(dataset[190]['label'])
print(dataset[190]['text'])

6
"What's the most interesting thing you've learned recently?"


In [84]:
import pandas as pd
df=pd.DataFrame(dataset)
df.to_csv("/kaggle/working/data.csv",index=False)

In [85]:
df_load=pd.read_csv("/kaggle/working/data.csv")
print(df_load)

                                                  text  label
0    Could you define the term "quantum entanglement"?      0
1    What does the term "photosynthesis" mean in th...      0
2    Can you define the concept of "natural selecti...      0
3      What does "homeostasis" refer to in physiology?      0
4    Could you define the term "isotope" in chemistry?      0
..                                                 ...    ...
551            What's a good way to start the morning?      6
552                     Why do people enjoy traveling?      6
553                         Let's talk about football.      6
554                  What are some useful life skills?      6
555                    Can you guess what mood I'm in?      6

[556 rows x 2 columns]


In [86]:
from sklearn.model_selection import train_test_split

train_text,val_text,train_label,val_label=train_test_split(df_load['text'],df_load['label'],test_size=0.2)

In [87]:
from transformers import BertTokenizer

intent_tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

def encode_texts(texts):
    return intent_tokenizer(list(texts),padding="max_length",truncation=True,max_length=128)

In [88]:
train_encodings=encode_texts(train_text)
val_encodings=encode_texts(val_text)

In [90]:
import torch
from torch.utils.data import Dataset

class IntentDataset(Dataset):
    def __init__(self,encodings,labels):
        self.encodings=encodings
        self.labels=labels

    def __getitem__(self,idx):
        item={key: torch.tensor(val[idx]) for key,val in self.encodings.items()}
        item['labels']=torch.tensor(self.labels[idx],dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

In [91]:
train_dataset=IntentDataset(train_encodings,train_label.to_list())
val_dataset=IntentDataset(val_encodings,val_label.to_list())

In [92]:
from transformers import TrainingArguments,Trainer,BertForSequenceClassification

intent_model=BertForSequenceClassification.from_pretrained("bert-base-uncased",num_labels=7)
intent_model.to(device)

training_args=TrainingArguments(
    output_dir="/kaggle/working",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
)

trainer=Trainer(
    model=intent_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packag

Epoch,Training Loss,Validation Loss
1,No log,1.977742
2,No log,0.406701
3,No log,0.124610
4,No log,0.107664
5,No log,0.111275
6,No log,0.106912
7,No log,0.113501
8,No log,0.112867
9,No log,0.113878
10,No log,0.113781


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


TrainOutput(global_step=280, training_loss=0.3834646224975586, metrics={'train_runtime': 83.5736, 'train_samples_per_second': 53.127, 'train_steps_per_second': 3.35, 'total_flos': 292066382592000.0, 'train_loss': 0.3834646224975586, 'epoch': 10.0})

In [93]:
print(set(val_label.to_list()))

{0, 1, 2, 3, 4, 5, 6}


In [135]:
intents={
    0: "Definition Only",
    1: "Explanation",
    2: "Give Examples",
    3: "Comparison",
    4: "Categorization",
    5: "Give Practice Questions",
    6: "General Chat/Other(NOT RELATED to SCIENCE)"
}

def predict_intent(query):
    tokenized_query=intent_tokenizer(query,return_tensors="pt",padding=True,truncation=True).to(device)
    with torch.no_grad():
        logits=intent_model(**tokenized_query).logits
    predicted_class_id=torch.argmax(logits,dim=1).item()
    return intents[predicted_class_id]

In [152]:
print(predict_intent("how are you??"))

General Chat/Other(NOT RELATED to SCIENCE)


In [153]:
print(predict_intent("explain what are cell organelles"))

Explanation


In [154]:
print(predict_intent("provide an example on cell organelles"))

Give Examples


In [155]:
print(predict_intent("Do you have any pets?"))

General Chat/Other(NOT RELATED to SCIENCE)


In [156]:
print(predict_intent("Give some questions on Oncology"))

Give Practice Questions


In [157]:
print(predict_intent("what is 'Cell membrane'?"))

Definition Only


In [158]:
print(predict_intent("How does a cell works?"))

Explanation


In [159]:
print(predict_intent("Categorize different types of cells"))

Categorization


In [124]:
def rewrite_query(query,chat_history,intent):
    if not chat_history:
        return query
        
    prompt = f"""
    You are an intelligent query rewriting assistant.
    
    Your task is to convert a follow-up query into a standalone query.
    
    STRICT RULES:
    - REWRITE QUERY ONLY IF IT IS NOT CLEAR ELSE RETURN THE SAME QUERY!!
    - Preserve BOTH: TOPICS and INTENT.
    - Get TOPICS from chat_history
    - Keep it concise.
    - Return ONLY the final standalone query.
    
    TOPICS:
    {chat_history}

    INTENT:
    {intent}
    
    Follow-up Query:
    {query}
    
    Rewritten Query:
    """

    tokenized_prompt=gemma_tokenizer(prompt,return_tensors="pt",truncation=True)
    input_len=tokenized_prompt["input_ids"].shape[1]
    output=gemma_model.generate(**tokenized_prompt,max_new_tokens=50)
    rewritten_query=gemma_tokenizer.decode(output[0][input_len:],skip_special_tokens=True)
    rewritten_query=rewritten_query.split('\n')[0]
    rewritten_query=" ".join(rewritten_query.split())

    return rewritten_query

In [125]:
def generate_answer(query,retrieved_texts):
    context=" ".join(retrieved_texts)
    input_text = f"""
    You are an educational assistant.

    TASK: You must RESPOND TO QUERY.

    RULES:
    - Follow the TASK strictly.
    - Use ONLY the context.
    - Do NOT repeat the same idea.
    - Do not add unrelated information.

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    inputs=gemma_tokenizer(input_text,return_tensors="pt",truncation=True)
    input_len=inputs["input_ids"].shape[1]
    outputs=gemma_model.generate(
        **inputs, 
        max_new_tokens=250,
        # min_new_tokens=40,
        repetition_penalty=1.5,
        # num_beams=4,
        # length_penalty=1.0,
        # no_repeat_ngram_size=2
        # do_sample=True,
        # temperature=0.7,
        # top_p=0.9
    )
    answer=gemma_tokenizer.decode(outputs[0][input_len:],skip_special_tokens=True,clean_up_tokenization_spaces=True) # outputs is 2D

    return answer

In [111]:
pip install keybert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [112]:
from keybert import KeyBERT

kb_model=KeyBERT("all-MiniLM-L6-v2")

def extract_keywords(query):
    keywords=kb_model.extract_keywords(query,keyphrase_ngram_range=(1,2),stop_words="english",top_n=3)
    return rank_keywords(keywords,query)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [126]:
def rank_keywords(keywords,query):
    ranked_list=[]
    for kw,score in keywords:
        if " " in kw:  #kw is multi-word
            score+=0.1
        if query.lower().endswith(kw.lower()):  # kw is the ending word
            score+=0.2
        ranked_list.append((kw,score))

    ranked_list.sort(key=lambda x:x[1])
    return [item[0] for item in ranked_list]

In [114]:
print(extract_keywords("what is a cell organelle?"))

[('cell', 0.7131), ('organelle', 0.8363), ('cell organelle', 1.0192)]
['cell', 'organelle', 'cell organelle']


In [115]:
print(extract_keywords("Give an example of cell organelle"))

[('example cell', 0.7924), ('organelle', 1.0101), ('cell organelle', 1.175)]
['example cell', 'organelle', 'cell organelle']


In [151]:
chat_history=[]

def edu_chatbot(query):
    global chat_history

    query_intent=predict_intent(query)
    chat_history.append(f"User: {",".join(extract_keywords(query))}")
    rewritten_query=rewrite_query(query,chat_history,query_intent)
    retrieved_texts=retrieve(rewritten_query,5)
    answer=generate_answer(rewritten_query,retrieved_texts)
    chat_history.append(f"Bot: {",".join(extract_keywords(answer))}")

    chat_history=chat_history[-4:]

    return answer

In [134]:
print(next(intent_model.parameters()).device)
print(next(gemma_model.parameters()).device)

cuda:0
cuda:0


In [160]:
print(edu_chatbot("Hey! How're you?"))

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


As a large language model I do not have feelings or experiences like humans do so "how" am i is not applicable to me





In [161]:
print(edu_chatbot("what are cell organelles?"))

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Organelles ("little organs")  are specialized compartments or sacs found within some kinds of cells





In [162]:
print(edu_chatbot("Explain it"))

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Organelles are specialized compartments or "mini-organs" found within certain kinds of cells known as eukaryotes. They're surrounded by membranes and act like tiny factories carrying out essential tasks crucial for keeping the entire cell functioning properly





In [163]:
print(edu_chatbot("give some examples"))

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


Nucleus,mitochondria  and ribosomes





In [164]:
print(edu_chatbot("Please give some practice questions on this?"))

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


What does lysosome do? What happens if it malfunctions?





In [165]:
print(edu_chatbot("Do you have any hobby?"))

/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2637: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cpu, whereas the model is on cuda. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cuda') before running `.generate()`.
  warnings.warn(


As I am designed only to process text input and output,  I do not engage in hobbies like humans do.



